# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [4]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [5]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [6]:
# set up environment
load_dotenv(override=True)

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [7]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [8]:
# prompts

system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
user_prompt = "Please give a detailed explanation to the following question: " + question

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
# Get gpt-4o-mini to answer, with streaming

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True,
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    response = response.replace("```", "").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

In [9]:
# Get Llama 3.2 to answer

response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
display(Markdown(response.choices[0].message.content))

This piece of code uses a concept called "yield from" in Python, which is a part of the syntax introduced in Python 3.3. Let's break down the code step by step.

### The Code
```python
yield from {book.get("author") for book in books if book.get("author")}
```
This line of code uses a few Python features, including a dictionary comprehension, a generator expression, and the "yield from" keyword.

Here's what each part does:

* `books` is presumably a list or similar iterable that contains dictionaries, possibly representing books with their metadata.
* The dictionary comprehension `(book.get("author") for book in books if book.get("author"))` is used to create an iterator that yields the "author" values from the books that also have an "author" key. The `if book.get("author")` part filters out books without an author.

* The `yield from` part takes this iterator (which returns only values and not a generator expression itself) and yields all the values yielded by this iterator.

### Why `yield from`?
The reason for using `yield from` here is that it allows us to take the output of the dictionary comprehension (which is a list of authors) and turn it into a generator that produces only the values (in this case, author names). Without `yield from`, we would get a list of authors as output from the dictionary comprehension, but this can be inefficient if we don't need to process all the values at once.

By using `yield from` in combination with the dictionary comprehension, we get an iterator that produces each author one at a time, which can be more memory-and-time efficient when dealing with large datasets.

### Why not direct iteration?
The expression `(book.get("author") for book in books if book.get("author"))` creates a generator expression, which returns an iterator over the "author" values directly. This would also work fine here and avoid the use of `yield from`. The use of `yield from` is more about creating an iterator inside another iterator rather than directly iterating.

Here's the code without `yield from`, for better clarity:

```python
for book in books:
    if book.get("author"):
        for author in book.get("author"):
            yield author
```
However, the original version with `yield from` serves the same purpose in a more concise way.